In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Create Silver database if needed
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Customers
bronze_customers = spark.table("bronze_customers")

silver_customers = (bronze_customers
    .dropDuplicates(["customer_id"])
    .withColumnRenamed("customer_zip_code_prefix", "customer_zip_code")
    .withColumn("customer_zip_code", col("customer_zip_code").cast("int"))
    .withColumn("silver_load_timestamp", current_timestamp())
    .withColumn("is_active", lit(True)))

silver_customers.write.mode("overwrite").format("delta").saveAsTable("silver.customers")


StatementMeta(, 318a92cf-66ea-4f62-b698-1719f88e92ac, 3, Finished, Available, Finished, False)

In [2]:
# Orders
bronze_orders = spark.table("bronze_orders")

silver_orders = (bronze_orders
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date"))
    .withColumn("order_year", year("order_purchase_timestamp"))
    .withColumn("order_month", month("order_purchase_timestamp"))
    .withColumn("silver_load_timestamp", current_timestamp()))

silver_orders.write.mode("overwrite").format("delta").saveAsTable("silver.orders")

StatementMeta(, 318a92cf-66ea-4f62-b698-1719f88e92ac, 4, Finished, Available, Finished, False)

In [3]:
# Products
bronze_products = spark.table("bronze_products")
bronze_category = spark.table("bronze_category_translation")

silver_products = (bronze_products.join(bronze_category.select(
            "product_category_name",
            "product_category_name_english"),
        "product_category_name",
        "left")
    .dropDuplicates(["product_id"])
    .withColumn("product_weight_g", col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", col("product_width_cm").cast("double"))
    .withColumn("silver_load_timestamp", current_timestamp())
)

silver_products.write.mode("overwrite").format("delta").saveAsTable("silver.products")

StatementMeta(, 318a92cf-66ea-4f62-b698-1719f88e92ac, 5, Finished, Available, Finished, False)

In [4]:
# Sellers

bronze_sellers = spark.table("bronze_sellers")

silver_sellers = (bronze_sellers
    .dropDuplicates(["seller_id"])
    .withColumnRenamed("seller_zip_code_prefix", "seller_zip_code")
    .withColumn("seller_zip_code", col("seller_zip_code").cast("int"))
    .withColumn("silver_load_timestamp", current_timestamp()))

silver_sellers.write.mode("overwrite").format("delta").saveAsTable("silver.sellers")

StatementMeta(, 318a92cf-66ea-4f62-b698-1719f88e92ac, 6, Finished, Available, Finished, False)

In [5]:
# Order Items
bronze_order_items = spark.table("bronze_order_items")

silver_order_items = (bronze_order_items
    .dropDuplicates(["order_id", "order_item_id", "product_id", "seller_id"])
    .withColumn("shipping_limit_date", to_timestamp("shipping_limit_date"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
    .withColumn("silver_load_timestamp", current_timestamp()))

silver_order_items.write.mode("overwrite").format("delta").saveAsTable("silver.order_items")

StatementMeta(, 318a92cf-66ea-4f62-b698-1719f88e92ac, 7, Finished, Available, Finished, False)

In [6]:
# Payments table
bronze_payments = spark.table("bronze_payments")

silver_payments = (bronze_payments
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn("payment_sequential", col("payment_sequential").cast("int"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
    .withColumn("payment_value", col("payment_value").cast("double"))
    .withColumn("silver_load_timestamp", current_timestamp()))

silver_payments.write.mode("overwrite").format("delta").saveAsTable("silver.payments")


print("Silver tables created successfully.")

StatementMeta(, 318a92cf-66ea-4f62-b698-1719f88e92ac, 8, Finished, Available, Finished, False)

Silver tables created successfully.
